# ✅ Clinical NLP Theme Mining for Surgical Burden & Case Complexity (2022–2025)
### Linking Diagnosis–Procedure Text Themes to Age, Sex, Anaesthesia, Urgency (EM/EL), and Specialty

---

## 📌 Why This Notebook Exists

In surgical theatre datasets, a large part of clinical meaning is stored in **free text**, mainly:

- Diagnosis descriptions
- Procedure / operation names

But hospital text is messy and inconsistent:

- abbreviations (e.g., SDH, ICH, VPS, CPD)
- shorthand and mixed spelling
- punctuation and incomplete wording
- different terms that mean the same thing

This makes it hard to identify true theatre burden drivers using standard tabulations alone.

Natural Language Processing (NLP) allows us to transform free text into structured intelligence—discovering hidden patterns in surgical workload and case complexity.

---

## ✅ What Makes This Notebook Different from the Previous Surgical Trends Notebook

In earlier notebooks, we analyzed overall surgical trends and distributions (e.g., overall emergency load, specialty volumes, etc.).

In **this notebook**, we will not re-do general descriptive summaries.

Instead, we use a **theme-first approach**:

1. We learn themes from the combined clinical text (Diagnosis + Procedure)
2. Then we interpret each theme using the contextual variables:
   - Age
   - Sex
   - Anaesthesia type
   - Emergency vs Elective (Urgency)
   - Specialty

**Key rule:**  
These variables will always be interpreted **in correspondence with diagnosis–procedure themes**, not in isolation.

---

## 🎯 Main Goal

To build an end-to-end Clinical NLP pipeline that:

✅ learns hidden diagnosis–procedure burden themes  
✅ groups similar cases into clusters (complexity-like patterns)  
✅ discovers clinical topics dominating theatre workload  
✅ mines diagnosis → procedure linkages  
✅ translates themes into actionable intelligence by stratifying them by:
- Age
- Sex
- Anaesthesia type
- Emergency vs Elective (Urgency)
- Specialty

---

## ✅ Learning Objectives (What I Will Learn in This Notebook)

By completing this notebook, I will learn real-world Clinical NLP skills used in healthcare analytics:

### 1) Clinical Text Preprocessing
- cleaning real hospital text
- handling abbreviations and shorthand
- standardizing spelling and duplicate meanings

### 2) Tokenization & Clinical Vocabulary Building
- extracting medical tokens
- stopwords and lemmatization (as appropriate)
- n-grams (unigrams + bigrams)

### 3) TF-IDF Feature Engineering
- converting text into numeric vectors
- making diagnosis + procedure machine-readable

### 4) Unsupervised Theme Discovery (Clustering)
- KMeans clustering using cosine similarity
- interpreting clusters clinically (theme naming)

### 5) Topic Modeling (LDA)
- automatically uncovering latent clinical themes
- interpreting topic word sets into meaningful clinical burden drivers

### 6) Diagnosis–Procedure Pair Mining
- identifying procedures most linked to diagnoses
- generating theatre burden intelligence tables

### 7) Theme-Based Stratified Intelligence (The Most Important Part)
For each discovered theme/cluster/topic, I will be able to answer:

- Which age groups dominate this theme?
- Is this theme more common in males or females?
- What anaesthesia type is most used in this theme?
- Is the theme mainly emergency-driven or elective?
- Which specialty contributes most to this theme?

This turns NLP results into theatre planning and complexity intelligence.

---

## 🏥 Dataset Description (2022–2025 Surgical Master)

This notebook uses the surgical theatre dataset covering:

- Years: **2022 to 2025**
- Each row represents one surgical case
- Key fields used here include:
  - Diagnosis (free text)
  - Operations / Procedures (free text)
  - Age
  - Sex
  - Anaesthesia type
  - Urgency (Emergency vs Elective)
  - Specialty

---

## ✅ Deliverables (What We Will Produce)

By the end of this notebook, we will produce:

- Top NLP-derived surgical burden themes (2022–2025)
- Theme-level emergency vs elective drivers
- Specialty contributions per theme
- Age and sex burden profiles per theme
- Anaesthesia utilization patterns by theme
- Diagnosis → procedure linkage intelligence
- Executive interpretation and planning recommendations

---

## ✅ Notebook Roadmap (Theme-First Workflow)

1. Load and validate dataset fields (ensure case-level correspondence)
2. Create combined clinical case text: **Diagnosis + Procedure**
3. Clean and normalize clinical text (abbreviations, spelling, duplicates)
4. Tokenization and vocabulary exploration
5. TF-IDF vectorization of combined case text
6. Theme clustering (KMeans) + clinical interpretation of clusters
7. Topic modeling (LDA) + clinical interpretation of topics
8. Diagnosis–procedure pair mining (linkage intelligence)
9. Theme-based stratification:
   - Age distribution per theme
   - Sex distribution per theme
   - Anaesthesia type per theme
   - Emergency vs elective split per theme
   - Specialty burden per theme
10. Executive outputs and recommendations

---

**Next, we will load the dataset and confirm the exact column names required for theme-based analysis.**


# ✅ Section 1: Data Loading & Case-Level Field Validation

---

## 🎯 Purpose of This Section

Before any NLP modeling, we must first ensure that our dataset contains all the required
case-level variables **in correspondence** with each diagnosis and procedure.

This is important because:

- Each row represents one surgical case  
- Diagnosis and procedure text must remain linked to:
  - Age  
  - Sex  
  - Anaesthesia type  
  - Emergency vs Elective status  
  - Specialty  

In this notebook, these variables will not be analyzed separately, but will be interpreted
**within NLP-derived clinical themes**.

---

## ✅ Step 1.1: Load the Surgical Master Dataset (2022–2025)

We begin by importing the dataset and confirming its structure.


In [1]:
import pandas as pd

# Load the Surgical Master Dataset
df = pd.read_excel("SURGICAL_MASTER_2022_2025.xlsx")

# Confirm dataset shape
print("Dataset Shape:", df.shape)

# Preview the first 5 rows
df.head()

Dataset Shape: (62459, 15)


,MONTH,AGE,SEX,TYPE_OF_SURGERY,SPECIALTY,DIAGNOSIS,OPERATIONS,ANESTHESIA,YEAR,MONTH_NAME,YEAR_MONTH,AGE_NUM,AGE_YEARS,AGE_GROUP,ANESTHESIA_IMPUTED
0,2022-01-01,27,MALE,EL,OPHTHA,CORNEAL LACERATION,CORNEAL REPAIR,LA,2022,January,2022-01,27.0,27.0,19–35,False
1,2022-01-01,9,MALE,EL,OPHTHA,CORNEAL PERFORATION,CORNEAL REPAIR,GA,2022,January,2022-01,9.0,9.0,6–12,False
2,2022-01-01,32,FEMALE,EM,GYNAE,RIGHT ECTOPIC PREGNANCY,EXPLORATORY LAPAROTOMY/SALPINGECTOMY,GA,2022,January,2022-01,32.0,32.0,19–35,False
3,2022-01-01,43,MALE,EM,GEN. SURGERY,MULTIPLE FACIAL LACERATION,DEBRIDEMENT/STITCHING,GA,2022,January,2022-01,43.0,43.0,36–60,False
4,2022-01-01,22,MALE,EM,GEN. SURGERY,COMPLICATED APPENDICITS,APPENDICITS,GA,2022,January,2022-01,22.0,22.0,19–35,False


## Dataset Overview

This dataset contains theatre operation records spanning **2022–2025**.
Each row represents a single surgical case and includes both structured
and unstructured clinical information.

Key attributes include:
- Patient demographics (age, sex)
- Urgency of surgery (emergency vs elective)
- Surgical specialty
- Diagnosis text (free-text clinical descriptions)
- Procedure/operation text (free-text)
- Anaesthesia type

In total, the dataset captures **over 62,000 surgical cases**, making it
suitable for large-scale Clinical NLP analysis focused on understanding
surgical burden, complexity, and resource utilization.


# ✅ Section 2: Building the Combined Clinical Case Narrative

---

## 🎯 Purpose of This Section

Clinical NLP requires a text representation of each surgical case.

In theatre datasets, the full clinical meaning of a case is not contained in diagnosis alone
or procedure alone, but in the **combination** of both.

For example:

- Diagnosis: *Subdural Hematoma*  
- Procedure: *Burr Hole Drainage*  

Together define the true burden and complexity of the case:

> "subdural hematoma burr hole drainage"

Therefore, in this notebook we construct a single combined clinical narrative for each case:

✅ Diagnosis + Procedure → One text field

This will become the foundation for:

- clinical text cleaning  
- tokenization  
- TF-IDF feature building  
- clustering into surgical themes  
- topic modeling of theatre burden patterns  

---

## ✅ Step 2.1: Create the Combined Case Text Field

We will create a new column called:

`case_text_raw`

that merges diagnosis and operation text for every surgical case.


In [2]:
# Create combined clinical case narrative: Diagnosis + Procedure
df["case_text_raw"] = (
    df["DIAGNOSIS"].fillna("") + " " +
    df["OPERATIONS"].fillna("")
)

# Preview to confirm it worked
df[["DIAGNOSIS", "OPERATIONS", "case_text_raw"]].head()


,DIAGNOSIS,OPERATIONS,case_text_raw
0,CORNEAL LACERATION,CORNEAL REPAIR,CORNEAL LACERATION CORNEAL REPAIR
1,CORNEAL PERFORATION,CORNEAL REPAIR,CORNEAL PERFORATION CORNEAL REPAIR
2,RIGHT ECTOPIC PREGNANCY,EXPLORATORY LAPAROTOMY/SALPINGECTOMY,RIGHT ECTOPIC PREGNANCY EXPLORATORY LAPAROTOMY...
3,MULTIPLE FACIAL LACERATION,DEBRIDEMENT/STITCHING,MULTIPLE FACIAL LACERATION DEBRIDEMENT/STITCHING
4,COMPLICATED APPENDICITS,APPENDICITS,COMPLICATED APPENDICITS APPENDICITS


# ✅ Section 3: Clinical Text Cleaning and Normalization

---

## 🎯 Purpose of This Section

Hospital diagnosis and procedure text is not clean English.

Clinical theatre narratives often contain:

- abbreviations (e.g., SDH, ICH, VPS, CPD)
- clinically meaningful symbols (e.g., `#` meaning fracture)
- shorthand expressions (e.g., `2/2` meaning due to)
- punctuation separators (e.g., "/", "&")
- inconsistent capitalization and spacing
- spelling variation and duplicate clinical meanings

Before applying TF-IDF, clustering, or topic modeling, we must build a robust
clinical text preprocessing pipeline.

---

## ✅ Key Clinical NLP Principle

In healthcare NLP, text cleaning is not just removing symbols.

Instead, we must first:

✅ translate clinically meaningful shorthand into standardized medical words

Examples:

- `#` → fracture  
- `2/2` → due to  
- `d/t` → due to  

Only after preserving meaning do we remove remaining punctuation noise.

---

## ✅ Step 3.1: Build a Clinical Cleaning Function

In this first cleaning step, we will:

- convert text to lowercase
- normalize important clinical symbols and shorthand
- remove unnecessary punctuation
- remove extra spacing

This produces a cleaned baseline clinical text field called:

`case_text_clean`

Later, we will extend this pipeline further with:

- abbreviation expansion dictionaries  
- synonym harmonization  
- advanced clinical vocabulary standardization  

---

## ✅ Output of This Section

At the end of Section 3.1, every surgical case will have:

- a raw combined narrative: `case_text_raw`
- a cleaned standardized narrative: `case_text_clean`

This cleaned text will serve as the foundation for all NLP modeling steps that follow.


In [3]:
import re

def clean_clinical_text(text):
    text = str(text).lower()

    # ---- Normalize clinically meaningful shorthand/symbols FIRST ----
    text = text.replace("#", " fracture ")

    # "due to" shorthand forms
    text = re.sub(r"\b2/2\b", " due to ", text)
    text = re.sub(r"\bd/t\b", " due to ", text)
    text = re.sub(r"\bdt\b", " due to ", text)

    # common separators
    text = text.replace("&", " and ")
    text = text.replace("/", " ")

    # ---- Remove remaining punctuation ----
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # ---- Remove extra spaces ----
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [4]:
df["case_text_clean"] = df["case_text_raw"].apply(clean_clinical_text)

df[["case_text_raw", "case_text_clean"]].head()


,case_text_raw,case_text_clean
0,CORNEAL LACERATION CORNEAL REPAIR,corneal laceration corneal repair
1,CORNEAL PERFORATION CORNEAL REPAIR,corneal perforation corneal repair
2,RIGHT ECTOPIC PREGNANCY EXPLORATORY LAPAROTOMY...,right ectopic pregnancy exploratory laparotomy...
3,MULTIPLE FACIAL LACERATION DEBRIDEMENT/STITCHING,multiple facial laceration debridement stitching
4,COMPLICATED APPENDICITS APPENDICITS,complicated appendicits appendicits


## ✅ Step 3.2: Clinical Abbreviation Expansion Strategy

---

### 🎯 Why Abbreviation Handling Matters

After basic cleaning, clinical case narratives still contain many abbreviations and shorthand
forms that differ across specialties.

Examples include:

- SDH → Subdural Hematoma  
- VPS → Ventriculo-Peritoneal Shunt  
- BPH → Benign Prostatic Hyperplasia  
- CPD → Context-dependent abbreviation  

If abbreviations are not handled correctly, NLP models will treat them as unrelated terms,
reducing theme clustering quality.

---

### ⚠️ Important Clinical NLP Challenge

Abbreviations are often **specialty-dependent**.

For example:

- CPD may mean *cephalopelvic disproportion* in Obstetrics  
- The same token may carry different meaning in another specialty  

Therefore, abbreviation expansion cannot be done blindly.

---

### ✅ Our Approach in This Notebook

Instead of guessing abbreviations manually, we will:

1. Discover abbreviation candidates directly from the dataset  
2. Rank them by frequency  
3. Inspect specialty usage patterns  
4. Expand only:
   - safe abbreviations globally  
   - ambiguous abbreviations using specialty-aware rules  

This ensures that abbreviation normalization is clinically valid and improves NLP modeling.


## ✅ Step 3.3: Abbreviation Discovery (Data-Driven)

---

### 🎯 Purpose

Clinical abbreviations vary widely across specialties and even across clinicians.
Therefore, instead of manually listing abbreviations, we first discover them directly
from the dataset.

This step answers:

- What abbreviations exist in the case narratives?
- How frequent are they?
- Which specialties use them most?

This helps us build a **data-driven abbreviation dictionary**, and also identify
**ambiguous abbreviations** that must be expanded using specialty context
(e.g., CPD in OBS/Gyn).

---

### ✅ Output

At the end of this step we will produce:

1) A ranked list of abbreviation candidates across the whole dataset  
2) A specialty-specific list (e.g., Neurosurgery, Obs/Gyn, Orthopaedics)  
3) A shortlist of abbreviations to expand safely vs those needing context rules


In [5]:
from collections import Counter
import re

# We'll use the ORIGINAL combined case text for abbreviation detection (keeps uppercase patterns)
text_series = df["case_text_raw"].fillna("").astype(str)

# Tokenize while preserving simple medical-like tokens (letters/numbers/#/slashes)
# This is for discovery only (we are not modeling yet)
tokens = []
for t in text_series:
    tokens.extend(re.findall(r"[A-Za-z0-9/#]+", t))

token_counts = Counter(tokens)

# Candidate abbreviations: mostly uppercase, short, not pure numbers
abbrev_candidates = [
    (tok, cnt) for tok, cnt in token_counts.items()
    if tok.isupper() and 2 <= len(tok) <= 6 and not tok.isdigit()
]

abbrev_candidates_sorted = sorted(abbrev_candidates, key=lambda x: x[1], reverse=True)

# Show top 40 abbreviation candidates in the dataset
abbrev_candidates_sorted[:40]


[('RIGHT', 8569),
 ('LEFT', 8262),
 ('AND', 5163),
 ('OF', 4556),
 ('WITH', 3948),
 ('OPEN', 3549),
 ('EYE', 3507),
 ('SCAR', 3504),
 ('FETAL', 3213),
 ('NON', 2951),
 ('STATUS', 2648),
 ('REPAIR', 2640),
 ('IN', 2199),
 ('FEMUR', 1932),
 ('KNEE', 1807),
 ('WOUND', 1594),
 ('TOTAL', 1461),
 ('BIOPSY', 1418),
 ('POST', 1418),
 ('LABOUR', 1406),
 ('HERNIA', 1387),
 ('BODY', 1349),
 ('INJURY', 1287),
 ('UNDER', 1279),
 ('LENS', 1274),
 ('TEAR', 1261),
 ('MASS', 1238),
 ('CANCER', 1203),
 ('TO', 1155),
 ('SMALL', 1125),
 ('DISTAL', 1119),
 ('ACUTE', 1118),
 ('TUMOR', 1109),
 ('HIP', 1076),
 ('TIBIA', 1055),
 ('SHUNT', 979),
 ('SKIN', 952),
 ('BREECH', 928),
 ('GRAFT', 928),
 ('FOOT', 903)]

In [6]:
text_series = df["case_text_raw"].fillna("").astype(str)

tokens = []
for t in text_series:
    tokens.extend(re.findall(r"[A-Za-z0-9]+", t))

token_counts = Counter(tokens)

# ✅ Abbreviation candidates = short tokens only (2–6 chars)
abbrev_candidates = [
    (tok, cnt) for tok, cnt in token_counts.items()
    if 2 <= len(tok) <= 6
]

# Sort by frequency
abbrev_candidates_sorted = sorted(abbrev_candidates, key=lambda x: x[1], reverse=True)

# Show top 50 short tokens
abbrev_candidates_sorted[:50]


[('RIGHT', 8775),
 ('LEFT', 8446),
 ('AND', 5164),
 ('OF', 4556),
 ('SCAR', 4253),
 ('WITH', 3949),
 ('OPEN', 3675),
 ('EYE', 3530),
 ('NON', 3241),
 ('FETAL', 3227),
 ('REPAIR', 3053),
 ('STATUS', 2988),
 ('IN', 2200),
 ('FEMUR', 1982),
 ('KNEE', 1837),
 ('BIOPSY', 1808),
 ('TIBIA', 1755),
 ('WOUND', 1653),
 ('POST', 1626),
 ('TOTAL', 1511),
 ('LABOUR', 1491),
 ('HERNIA', 1443),
 ('LENS', 1361),
 ('BODY', 1360),
 ('INJURY', 1352),
 ('TEAR', 1320),
 ('UNDER', 1279),
 ('MASS', 1279),
 ('CANCER', 1220),
 ('TUMOR', 1179),
 ('TO', 1155),
 ('DISTAL', 1146),
 ('ACUTE', 1145),
 ('SMALL', 1134),
 ('HIP', 1089),
 ('BREECH', 1076),
 ('SHUNT', 1013),
 ('SKIN', 1005),
 ('GRAFT', 968),
 ('PRE', 942),
 ('FOOT', 920),
 ('SEPTIC', 914),
 ('FIBULA', 904),
 ('SEVERE', 883),
 ('DUE', 873),
 ('NAIL', 828),
 ('TENDON', 803),
 ('TOLAC', 783),
 ('NECK', 772),
 ('AT', 730)]

In [7]:
# Strong "not an abbreviation" list (we can keep growing it)
not_abbrev = {
    "RIGHT","LEFT","AND","OF","WITH","OPEN","EYE","NON","FETAL","REPAIR","STATUS","IN",
    "FEMUR","KNEE","BIOPSY","TIBIA","WOUND","POST","TOTAL","LABOUR","HERNIA","LENS","BODY",
    "INJURY","TEAR","UNDER","MASS","CANCER","TUMOR","TO","DISTAL","ACUTE","SMALL","HIP",
    "BREECH","SHUNT","SKIN","GRAFT","PRE","FOOT","SEPTIC","FIBULA","SEVERE","DUE","NAIL",
    "TENDON","NECK","AT","FOR","ON","BY","FROM","NO","YES","YEAR","MONTH","DAY"
}

def find_abbrev_candidates(series, min_len=2, max_len=6, top_n=40):
    tokens = []
    for t in series.fillna("").astype(str):
        tokens.extend(re.findall(r"[A-Za-z0-9]+", t))
    c = Counter(tokens)

    # Candidate logic:
    # 1) length 2-6
    # 2) not pure digits
    # 3) either contains a digit (T6, L4L5) OR is all letters and not a common word
    candidates = []
    for tok, cnt in c.items():
        if not (min_len <= len(tok) <= max_len):
            continue
        if tok.isdigit():
            continue

        tok_up = tok.upper()

        if tok_up in not_abbrev:
            continue

        has_digit = any(ch.isdigit() for ch in tok)
        is_letters_only = tok.isalpha()

        if has_digit or is_letters_only:
            candidates.append((tok_up, cnt))

    # merge same tokens different casing
    merged = Counter()
    for tok, cnt in candidates:
        merged[tok] += cnt

    return merged.most_common(top_n)

# Separate candidates
diag_abbrevs = find_abbrev_candidates(df["DIAGNOSIS"], top_n=40)
op_abbrevs   = find_abbrev_candidates(df["OPERATIONS"], top_n=40)

print("Top DIAGNOSIS abbreviation candidates:")
print(diag_abbrevs)

print("\nTop OPERATIONS abbreviation candidates:")
print(op_abbrevs)

Top DIAGNOSIS abbreviation candidates:
[('SCAR', 4243), ('TOLAC', 782), ('LEG', 631), ('SKULL', 611), ('TERM', 590), ('BURNS', 571), ('CYST', 560), ('FAILED', 554), ('HAND', 545), ('RT', 536), ('DEGREE', 532), ('TWIN', 518), ('PHASE', 485), ('LT', 485), ('LABOR', 484), ('CUT', 418), ('3RD', 415), ('ULCER', 412), ('PELVIC', 402), ('2ND', 401), ('STAGE', 387), ('FINGER', 372), ('CORD', 369), ('RADIUS', 351), ('BREAST', 330), ('BENIGN', 320), ('UPPER', 320), ('GOITRE', 309), ('ACTIVE', 308), ('NRFS', 298), ('SLEEP', 293), ('APNEA', 288), ('THE', 261), ('ULNA', 255), ('LOWER', 250), ('MITRAL', 246), ('ANKLE', 234), ('HEART', 231), ('LID', 226), ('CEPHA', 223)]

Top OPERATIONS abbreviation candidates:
[('SPLIT', 693), ('TUBE', 568), ('FUSION', 486), ('ABOVE', 371), ('CLOSED', 347), ('TUBAL', 338), ('FLAP', 330), ('BELOW', 320), ('DIRECT', 297), ('PLUS', 288), ('DRAIN', 270), ('CS', 266), ('2ND', 234), ('ORIF', 227), ('PLATE', 210), ('VENOUS', 204), ('VALVE', 203), ('RIGID', 199), ('BONE', 1

In [8]:
vowels = set("AEIOU")

def abbrev_like(tok):
    tok = tok.upper()
    # remove pure numbers
    if tok.isdigit():
        return False
    
    # keep if contains any digit (T6, L4L5, 3RD)
    if any(ch.isdigit() for ch in tok):
        return True
    
    # letters-only short tokens that look like abbreviations
    if tok.isalpha() and 2 <= len(tok) <= 6:
        # count vowels
        vcount = sum(1 for ch in tok if ch in vowels)
        # abbreviations often have 0-1 vowels (SDH, ICH, VPS, NRFS)
        return vcount <= 1
    
    return False

def find_abbrev_like(series, top_n=40):
    tokens = []
    for t in series.fillna("").astype(str):
        tokens.extend(re.findall(r"[A-Za-z0-9]+", t))
    c = Counter(tokens)

    # apply abbrev-like filter
    cand = Counter()
    for tok, cnt in c.items():
        if abbrev_like(tok):
            cand[tok.upper()] += cnt
    
    return cand.most_common(top_n)

diag_abbrev_like = find_abbrev_like(df["DIAGNOSIS"], top_n=40)
op_abbrev_like   = find_abbrev_like(df["OPERATIONS"], top_n=40)

print("DIAGNOSIS abbrev-like tokens:")
print(diag_abbrev_like)

print("\nOPERATIONS abbrev-like tokens:")
print(op_abbrev_like)


DIAGNOSIS abbrev-like tokens:
[('RIGHT', 6683), ('LEFT', 6436), ('SCAR', 4243), ('NON', 3197), ('WITH', 2960), ('IN', 2168), ('OF', 2000), ('POST', 1452), ('MASS', 1165), ('TO', 1122), ('BODY', 984), ('AND', 951), ('PRE', 940), ('AT', 702), ('NECK', 678), ('LEG', 631), ('SKULL', 611), ('TERM', 590), ('BURNS', 571), ('CYST', 560), ('HIP', 549), ('HAND', 545), ('RT', 536), ('TWIN', 518), ('LT', 485), ('CUT', 418), ('3RD', 415), ('2ND', 401), ('CORD', 369), ('NRFS', 298), ('THE', 261), ('LID', 226), ('LIMB', 222), ('BOTH', 222), ('SCALP', 219), ('ON', 216), ('THIGH', 215), ('1PREVIOUS', 202), ('END', 201), ('4TH', 189)]

OPERATIONS abbrev-like tokens:
[('AND', 4213), ('OF', 2556), ('RIGHT', 2092), ('LEFT', 2010), ('LENS', 1341), ('SMALL', 1010), ('WITH', 989), ('SKIN', 975), ('GRAFT', 929), ('SHUNT', 889), ('SPLIT', 693), ('HIP', 540), ('BODY', 376), ('FLAP', 330), ('PLUS', 288), ('CS', 266), ('2ND', 234), ('STENT', 184), ('1ST', 180), ('POST', 174), ('STUMP', 166), ('3RD', 159), ('SCREWS

### ✅ Observation from Abbreviation Discovery

Abbreviation mining shows that this dataset is largely composed of fully written
clinical terminology rather than heavy shorthand.

Only a small number of true abbreviations appear frequently (e.g., RT/LT, CS, ORIF, NRFS).

Therefore, the NLP preprocessing focus will prioritize:

- terminology harmonization  
- spelling normalization  
- consistent clinical vocabulary mapping  

rather than extensive abbreviation expansion.


## ✅ Step 3.4: Clinical Spelling and Terminology Harmonization

---

### 🎯 Purpose

Even when abbreviations are limited, clinical text still contains:

- spelling variants (appendicits vs appendicitis)
- duplicated terms (burr hole vs burrhole)
- directional shorthand (RT/LT)
- inconsistent phrasing across departments

To ensure accurate clustering and topic modeling, we harmonize the most common
clinical terminology variations using targeted replacements.

---

### ✅ Output

We create a final standardized modeling text field:

`case_text_final`

which will be used for TF-IDF, clustering, and topic modeling.


In [9]:
# Targeted clinical harmonization rules
harmonize_map = {
    "appendicits": "appendicitis",
    "cs": "caesarean section",
    "csection": "caesarean section",
    "rt": "right",
    "lt": "left",
    "burrhole": "burr hole"
}

def harmonize_terms(text):
    words = text.split()
    standardized = [harmonize_map.get(w, w) for w in words]
    return " ".join(standardized)

# Apply harmonization
df["case_text_final"] = df["case_text_clean"].apply(harmonize_terms)

# Compare examples
df[["case_text_clean", "case_text_final"]].head(10)


,case_text_clean,case_text_final
0,corneal laceration corneal repair,corneal laceration corneal repair
1,corneal perforation corneal repair,corneal perforation corneal repair
2,right ectopic pregnancy exploratory laparotomy...,right ectopic pregnancy exploratory laparotomy...
3,multiple facial laceration debridement stitching,multiple facial laceration debridement stitching
4,complicated appendicits appendicits,complicated appendicitis appendicitis
5,right hip fracture dislocation open reduction ...,right hip fracture dislocation open reduction ...
6,septic wound right knee debridement,septic wound right knee debridement
7,exposed ventriculo peritoneal shunt external v...,exposed ventriculo peritoneal shunt external v...
8,depressed skull fracture acute epidural hemato...,depressed skull fracture acute epidural hemato...
9,complex gastroschisis resection and anastomasi...,complex gastroschisis resection and anastomasi...


# ✅ Section 4: Tokenization and Clinical Vocabulary Exploration

---

## 🎯 Purpose of This Section

Now that we have constructed a cleaned and standardized clinical case narrative
(`case_text_final`), the next step is to explore the medical vocabulary contained
in the theatre workload.

Tokenization is the process of breaking clinical text into meaningful units (tokens),
such as:

- fracture  
- craniotomy  
- tumor  
- abscess  
- laparotomy  

This helps us understand the dominant clinical concepts before feature extraction.

---

## ✅ Step 4.1: Tokenize Clinical Case Narratives

In this step, we will:

- split clinical narratives into tokens
- build the most common clinical terms across all surgical cases
- begin identifying major surgical burden drivers

This vocabulary forms the foundation for TF-IDF representation in the next section.


In [10]:
# Tokenize case narratives into individual clinical words
df["tokens"] = df["case_text_final"].apply(lambda x: x.split())

# Build vocabulary frequency across all cases
all_tokens = [tok for row in df["tokens"] for tok in row]

token_counts = Counter(all_tokens)

# Display the top 20 most common clinical tokens
token_counts.most_common(20)


[('section', 12217),
 ('caesarean', 9778),
 ('right', 9445),
 ('left', 9033),
 ('fracture', 6970),
 ('debridement', 5413),
 ('and', 5347),
 ('of', 4556),
 ('scar', 4253),
 ('with', 3949),
 ('previous', 3913),
 ('open', 3675),
 ('eye', 3530),
 ('non', 3241),
 ('fetal', 3227),
 ('repair', 3053),
 ('reassuring', 2988),
 ('status', 2988),
 ('emergency', 2721),
 ('1', 2521)]

## ✅ Step 4.2: Removing Non-Informative Tokens (Stopwords)

---

### 🎯 Purpose

Raw token frequencies include both:

- meaningful clinical burden terms (fracture, tumor, debridement)
- non-informative common words (and, of, with, right, left)

Before feature extraction, we remove these high-frequency noise tokens so that NLP models
focus on the true clinical content.

In clinical NLP, we use two types of stopwords:

1. Standard English stopwords (and, of, with...)
2. Domain-specific theatre stopwords (right, left, previous...)

---

### ✅ Output

This step produces a refined vocabulary representing true clinical burden signals.


In [11]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Keep domain words like: open, non, status, section
clinical_stopwords = {"right", "left"}  # directional only for now

stopwords = set(ENGLISH_STOP_WORDS).union(clinical_stopwords)

df["tokens_refined"] = df["tokens"].apply(
    lambda toks: [t for t in toks if t not in stopwords and not t.isdigit()]
)

refined_tokens = [tok for row in df["tokens_refined"] for tok in row]
refined_counts = Counter(refined_tokens)

refined_counts.most_common(20)

[('section', 12217),
 ('caesarean', 9778),
 ('fracture', 6970),
 ('debridement', 5413),
 ('scar', 4253),
 ('previous', 3913),
 ('open', 3675),
 ('eye', 3530),
 ('non', 3241),
 ('fetal', 3227),
 ('repair', 3053),
 ('reassuring', 2988),
 ('status', 2988),
 ('emergency', 2721),
 ('reduction', 2464),
 ('fixation', 2424),
 ('cataract', 2222),
 ('laparotomy', 2171),
 ('femur', 1982),
 ('knee', 1837)]

### From Cleaned Clinical Text to Machine-Readable Features

After cleaning and normalizing diagnosis and procedure text,
the next step is to convert this unstructured clinical language
into numerical representations that machine learning algorithms
can process.

We use TF-IDF with clinical n-grams to capture both term importance
and contextual meaning across surgical cases.


# ✅ Section 5: TF-IDF Feature Engineering (Clinical Case Representation)

---

## 🎯 Purpose of This Section

Machines cannot directly understand clinical text.

To perform clustering or topic modeling, we must convert our cleaned surgical case narratives
into numeric features.

TF-IDF (Term Frequency–Inverse Document Frequency) is a core NLP method that transforms text into
a vector representation reflecting:

- how important a term is in a given case
- how unique that term is across all theatre cases

---

## ✅ Why We Use n-grams in Clinical NLP

Many surgical concepts occur as phrases rather than single words:

- open fracture  
- caesarean section  
- fetal status  
- burr hole  
- tumor excision  

Therefore, we use TF-IDF with **unigrams + bigrams** (1–2 grams)
to preserve clinically meaningful phrases.

---

## ✅ Output of This Section

We will produce:

- a TF-IDF matrix representing every surgical case numerically
- a vocabulary of important clinical burden terms and phrases

This matrix becomes the input for clustering and topic modeling.


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Vectorizer with clinical unigrams + bigrams
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2)
)

# Fit and transform the final cleaned clinical text
X_tfidf = tfidf.fit_transform(df["case_text_final"])

print("TF-IDF Matrix Shape:", X_tfidf.shape)


TF-IDF Matrix Shape: (62459, 3000)


In [13]:
# View first 20 TF-IDF feature names
tfidf.get_feature_names_out()[:20]


array(['10', '10th', '10th electroconvulsive', '10th session', '11th',
       '12th', '15', '1previous', '1previous scar', '1psc', '1st',
       '1st breech', '1st electro', '1st electroconvulsive',
       '1st session', '1st stage', '1st twin', '20', '28', '2nd'],
      dtype=object)

## ✅ Step 5.2: Removing Numeric and Ordinal Feature Noise

Clinical text often contains numeric artifacts such as:

- 1st, 2nd, 3rd stage
- session counts (10th session)
- pure numbers

These terms are not meaningful surgical burden concepts and may distort
theme clustering.

Therefore, we apply a token filter to remove numeric-only and ordinal patterns
before final TF-IDF modeling.


In [14]:
# Custom tokenizer filter: remove numeric + ordinal tokens
def clinical_token_filter(text):
    tokens = text.split()
    filtered = [
        t for t in tokens
        if not re.fullmatch(r"\d+", t)          # remove pure numbers
        and not re.fullmatch(r"\d+(st|nd|rd|th)", t)  # remove ordinals
    ]
    return " ".join(filtered)

# Apply filtering
df["case_text_model"] = df["case_text_final"].apply(clinical_token_filter)

# Re-run TF-IDF with cleaner text
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_tfidf = tfidf.fit_transform(df["case_text_model"])

print("Clean TF-IDF Matrix Shape:", X_tfidf.shape)

# View first 20 features again
tfidf.get_feature_names_out()[:20]


Clean TF-IDF Matrix Shape: (62459, 3000)


array(['1previous', '1previous scar', '1psc', '2previous',
       '2previous scar', '3a', '3b', '3c', 'abcess', 'abcess incision',
       'abdomen', 'abdominal', 'abdominal closure',
       'abdominal hysterectomy', 'abdominal injury', 'abdominal trauma',
       'abdominal uterine', 'abdominal wall', 'abnormal', 'abortion'],
      dtype=object)

## ✅ Step 5.3: Harmonizing Obstetric Scar History Terminology

In obstetrics, terms such as:

- 1 previous scar  
- 2 previous scars  
- PSC (previous scar)  

are clinically meaningful drivers of Caesarean Section complexity and workload.

Rather than removing these tokens, we standardize them into consistent phrases so that
repeat Caesarean burden forms a coherent NLP theme.

This improves clustering while preserving clinical meaning.


In [15]:
def harmonize_ob_history(text):
    text = text.replace("1psc", "previous scar")
    text = text.replace("2psc", "multiple previous scars")
    
    # Normalize patterns like "1previous scar"
    text = re.sub(r"\b1previous\b", "previous", text)
    text = re.sub(r"\b2previous\b", "multiple previous", text)
    text = re.sub(r"\b3previous\b", "multiple previous", text)
    
    return text

# Apply OB scar harmonization
df["case_text_model"] = df["case_text_model"].apply(harmonize_ob_history)

# Check the first 10 features again
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2)
)
X_tfidf = tfidf.fit_transform(df["case_text_model"])

print("TF-IDF Shape:", X_tfidf.shape)
tfidf.get_feature_names_out()[:25]

TF-IDF Shape: (62459, 3000)


array(['3a', '3b', '3c', 'abcess', 'abcess incision', 'abdomen',
       'abdominal', 'abdominal closure', 'abdominal hysterectomy',
       'abdominal injury', 'abdominal trauma', 'abdominal uterine',
       'abdominal wall', 'abnormal', 'abortion', 'abortion dilatation',
       'abruptio', 'abruptio caesarean', 'abruption',
       'abruption caesarean', 'abscess', 'abscess craniotomy',
       'abscess debridement', 'abscess drainage', 'abscess exploratory'],
      dtype=object)

### Identifying Surgical Burden Themes Using Unsupervised Learning

With each surgical case now represented as a numerical vector,
we apply unsupervised clustering to uncover dominant patterns
in surgical workload without pre-defined labels.

Clustering allows us to group cases that share similar
diagnosis–procedure language, revealing natural surgical burden themes.


# ✅ Section 6: Surgical Burden Theme Clustering (KMeans)

---

## 🎯 Purpose of This Section

We now use unsupervised learning to cluster surgical cases into hidden clinical themes
based on the combined diagnosis–procedure narrative.

Using the TF-IDF representation, we will:

- group similar cases together
- interpret each cluster clinically using its top terms
- label clusters into meaningful theatre burden themes

Examples of potential themes:

- Trauma fractures and fixation
- Obstetric emergencies and Caesarean burden
- Infection and debridement burden
- Tumors and excisions
- Neurosurgical hematomas and burr hole themes

---

## ✅ Step 6.1: Selecting a Reasonable Number of Clusters (k)

KMeans requires choosing the number of clusters (k).

We will use a simple elbow-style check on a sample to identify a reasonable range,
then proceed with an interpretable number of clusters for theme mining.

---

## ✅ Output of This Section

- cluster assignment for each case
- top clinical terms defining each cluster
- initial clinical interpretation of themes


In [16]:
import numpy as np
from sklearn.cluster import KMeans

# Sample for speed (keeps the notebook responsive)
np.random.seed(42)
sample_idx = np.random.choice(X_tfidf.shape[0], size=8000, replace=False)
X_sample = X_tfidf[sample_idx]

ks = [6, 8, 10, 12, 14, 16]
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sample)
    inertias.append(km.inertia_)
    print(f"k={k}, inertia={km.inertia_:.2f}")

inertias


k=6, inertia=7307.98
k=8, inertia=7114.10
k=10, inertia=6987.75
k=12, inertia=6912.03
k=14, inertia=6825.02
k=16, inertia=6736.63


[7307.978324329002,
 7114.102318644657,
 6987.752636374135,
 6912.02810143862,
 6825.022744039122,
 6736.6275690838565]

## ✅ Step 6.2: Fit KMeans Clustering on Full Theatre Dataset

We now fit the KMeans model using the selected number of clusters:

- **k = 8**

Each surgical case will be assigned a cluster label representing its
dominant diagnosis–procedure burden theme.

These clusters will form the foundation for:

- clinical theme interpretation
- emergency vs elective drivers
- specialty contributions
- age/sex/anaesthesia stratification


In [17]:
# Final number of clusters
k = 8

# Fit KMeans on full TF-IDF matrix
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["theme_cluster"] = kmeans.fit_predict(X_tfidf)

# Check cluster distribution
df["theme_cluster"].value_counts().sort_index()


0    36952
1     5651
2     1491
3     7918
4     1170
5     2989
6     4982
7     1306
Name: theme_cluster, dtype: int64

## ✅ Step 6.3: Clinical Interpretation of Clusters (Theme Naming)

KMeans clustering assigns each surgical case to a theme cluster, but clusters are only
useful if we interpret them clinically.

To understand what each cluster represents, we examine the top TF-IDF terms
that define each cluster center.

This allows us to translate cluster numbers into meaningful surgical burden themes such as:

- Obstetric emergencies
- Trauma fractures and fixation
- Infection and abscess burden
- Tumors and excisions
- Neurosurgical hematomas
- Ophthalmology repairs

---

### ✅ Output

For each cluster, we will display:

- Top defining clinical terms and phrases
- A provisional clinical theme label


In [18]:
# Feature names from TF-IDF
terms = tfidf.get_feature_names_out()

# Display top terms per cluster
for i in range(k):
    print(f"\n============================")
    print(f"Cluster {i} Top Clinical Terms:")
    
    center_terms = kmeans.cluster_centers_[i].argsort()[-12:][::-1]
    top_words = [terms[j] for j in center_terms]
    
    print(top_words)


Cluster 0 Top Clinical Terms:
['right', 'left', 'repair', 'laparotomy', 'excision', 'craniotomy', 'hematoma', 'biopsy', 'hernia', 'exploratory', 'exploratory laparotomy', 'subdural']

Cluster 1 Top Clinical Terms:
['fracture', 'open', 'fixation', 'reduction', 'open reduction', 'femur', 'internal', 'reduction internal', 'internal fixation', 'tibia', 'right', 'left']

Cluster 2 Top Clinical Terms:
['therapy', 'electroconvulsive', 'electroconvulsive therapy', 'schizophrenia', 'schizophrenia electroconvulsive', 'electro', 'convulsive', 'convulsive therapy', 'electro convulsive', 'session', 'therapy session', 'disorder']

Cluster 3 Top Clinical Terms:
['section', 'caesarean', 'caesarean section', 'scar', 'previous', 'previous scar', 'scar caesarean', 'labour', 'labour caesarean', 'arrested', 'emergency', 'breech']

Cluster 4 Top Clinical Terms:
['cataract', 'eye', 'lens', 'intraocular', 'small', 'small incision', 'cataract surgery', 'incision cataract', 'intraocular lens', 'surgery', 'inci

## ✅ Step 6.4: Theme-Based Patient and Theatre Resource Stratification

Once themes have been discovered, the next step is to interpret each theme in context.

For each surgical theme cluster, we will examine:

- Age group burden (who is most affected?)
- Sex distribution (male vs female load)
- Emergency vs elective drivers (urgency burden)
- Anaesthesia utilization (GA vs Spinal vs Local)
- Specialty contributions (department-level workload)

This converts NLP themes into actionable theatre complexity and planning intelligence.


In [19]:
# Summary of each theme cluster
theme_summary = df.groupby("theme_cluster").agg(
    cases=("theme_cluster", "count"),
    mean_age=("AGE_YEARS", "mean"),
    emergency_pct=("TYPE_OF_SURGERY", lambda x: (x=="EM").mean()*100),
    top_specialty=("SPECIALTY", lambda x: x.value_counts().idxmax()),
    top_anaesthesia=("ANESTHESIA", lambda x: x.value_counts().idxmax())
)

theme_summary.round(2)


,cases,mean_age,emergency_pct,top_specialty,top_anaesthesia
theme_cluster,,,,,
0,36952,32.61,42.33,GENERAL SURGERY,GA
1,5651,34.88,40.97,ORTHOPAEDIC,GA
2,1491,34.23,0.00,ECT,GA
3,7918,28.75,82.45,OBS,SA
4,1170,58.87,2.05,OPHTHAMOLOGY,LA
5,2989,26.90,98.96,OBSTETRICS,SA
6,4982,33.41,59.82,ORTHOPAEDIC,GA
7,1306,5.05,0.69,EAR NOSE AND THROAT,GA


### Discovering Latent Clinical Themes with Topic Modeling

While clustering groups similar cases, topic modeling focuses on
discovering recurring clinical concepts that cut across cases.

Latent Dirichlet Allocation (LDA) is used here to identify hidden
clinical themes that may not be immediately obvious from raw text
or clustering alone.


# ✅ Section 7: Topic Modeling for Hidden Clinical Theatre Burden Themes (LDA)

---

## 🎯 Purpose of This Section

While clustering assigns each surgical case into a single dominant theme,
clinical cases often contain multiple overlapping concepts.

For example:

- Trauma + infection  
- Tumor + craniotomy  
- Obstetric emergency + fetal distress  

Topic Modeling allows us to discover these **latent (hidden) clinical themes**
automatically.

---

## ✅ What Topic Modeling Does

Latent Dirichlet Allocation (LDA) identifies collections of words that frequently
appear together in surgical case narratives.

This produces:

- Topic 0: {fracture, femur, fixation, orif...}
- Topic 1: {abscess, debridement, wound...}
- Topic 2: {caesarean, fetal distress, emergency...}

Unlike clustering:

- One case can belong partly to multiple topics.

---

## ✅ Output of This Section

We will generate:

- A set of hidden theatre burden topics
- Top clinical words defining each topic
- Topic-level interpretation for executive planning


In [20]:
from sklearn.feature_extraction.text import CountVectorizer

# Count-based vectorizer with unigrams + bigrams
count_vec = CountVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_count = count_vec.fit_transform(df["case_text_model"])

print("Count Matrix Shape:", X_count.shape)


Count Matrix Shape: (62459, 3000)


## ✅ Step 7.2: Fit LDA Model to Discover Hidden Theatre Topics

We now fit a Latent Dirichlet Allocation (LDA) model to uncover hidden
clinical topics that repeatedly appear in surgical case narratives.

We begin with **8 topics** for interpretability and alignment with major
theatre burden categories.


In [21]:
from sklearn.decomposition import LatentDirichletAllocation

# Number of hidden topics
n_topics = 8

# Fit LDA model
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch"
)

lda_topics = lda.fit_transform(X_count)

print("LDA Topic Matrix Shape:", lda_topics.shape)


LDA Topic Matrix Shape: (62459, 8)


## ✅ Step 7.3: Topic Interpretation (Top Clinical Words per Topic)

After fitting LDA, each topic is defined by a set of words and phrases that
frequently occur together in theatre cases.

To interpret the topics clinically, we extract the top terms for each topic.

These topics represent hidden drivers of surgical burden and case complexity,
such as:

- trauma fixation
- fetal distress emergencies
- infection and debridement
- cataract surgery
- laparotomy and abdominal emergencies


In [22]:
# Feature names
topic_terms = count_vec.get_feature_names_out()

# Function to display top words per topic
def display_topics(model, feature_names, top_n=12):
    for topic_idx, topic in enumerate(model.components_):
        top_features = topic.argsort()[-top_n:][::-1]
        top_words = [feature_names[i] for i in top_features]
        
        print("\n===========================")
        print(f"Topic {topic_idx} Top Terms:")
        print(top_words)

# Display topics
display_topics(lda, topic_terms, top_n=12)



Topic 0 Top Terms:
['section', 'caesarean', 'caesarean section', 'fetal', 'non', 'reassuring', 'status', 'non reassuring', 'reassuring fetal', 'fetal status', 'status caesarean', 'emergency']

Topic 1 Top Terms:
['eye', 'cataract', 'left', 'right', 'left eye', 'right eye', 'lens', 'surgery', 'intraocular', 'corneal', 'intraocular lens', 'small']

Topic 2 Top Terms:
['excision', 'skin', 'graft', 'abdominal', 'abscess', 'thickness', 'biopsy', 'drainage', 'skin graft', 'appendicitis', 'thickness skin', 'split']

Topic 3 Top Terms:
['debridement', 'craniotomy', 'left', 'right', 'wound', 'amputation', 'injury', 'knee', 'foot', 'hematoma', 'tendon', 'fracture']

Topic 4 Top Terms:
['section', 'caesarean', 'caesarean section', 'scar', 'previous', 'previous scar', 'emergency', 'arrested', 'scar caesarean', 'therapy', 'electroconvulsive', 'electroconvulsive therapy']

Topic 5 Top Terms:
['hypertrophy', 'adenotonsillectomy', 'repair', 'tear', 'adenotonsillar', 'body', 'adenotonsillar hypertroph

## ✅ Step 7.4: Refining Topic Modeling for Better Specialty Separation

While the initial topic model provided strong high-level themes, some topics still
show overlap between unrelated specialties.

To improve interpretability and ensure broader specialty coverage across the theatre,
we increase the number of topics to **13**.

This refinement helps:

- separate obstetric sub-themes (scar vs fetal distress)
- isolate mental health procedural burden (ECT)
- distinguish neurosurgery from general surgery
- uncover additional specialty-driven workload themes


In [23]:
# Increase topic count for richer specialty coverage
n_topics_refined = 13

lda13 = LatentDirichletAllocation(
    n_components=n_topics_refined,
    random_state=42,
    learning_method="batch"
)

lda13_topics = lda13.fit_transform(X_count)

print("Refined LDA Topic Matrix Shape:", lda13_topics.shape)


Refined LDA Topic Matrix Shape: (62459, 13)


In [24]:
print("===== Top Clinical Terms per Refined Topic (13 Topics) =====")

display_topics(lda13, topic_terms, top_n=12)


===== Top Clinical Terms per Refined Topic (13 Topics) =====

Topic 0 Top Terms:
['section', 'caesarean', 'caesarean section', 'fetal', 'non', 'reassuring', 'status', 'non reassuring', 'reassuring fetal', 'fetal status', 'status caesarean', 'emergency']

Topic 1 Top Terms:
['eye', 'cataract', 'left', 'right', 'left eye', 'right eye', 'lens', 'surgery', 'intraocular', 'intraocular lens', 'corneal', 'small']

Topic 2 Top Terms:
['skin', 'graft', 'thickness', 'skin graft', 'drainage', 'appendicitis', 'thickness skin', 'split', 'split thickness', 'abscess', 'incision', 'incision drainage']

Topic 3 Top Terms:
['debridement', 'amputation', 'right', 'left', 'craniotomy', 'foot', 'knee', 'knee amputation', 'skull', 'septic', 'fracture', 'hematoma craniotomy']

Topic 4 Top Terms:
['section', 'caesarean', 'caesarean section', 'scar', 'previous', 'previous scar', 'emergency', 'cesarean', 'cesarean section', 'labour', 'scar caesarean', 'tolac']

Topic 5 Top Terms:
['hypertrophy', 'adenotonsillect

### Moving from Themes to Actionable Theatre Intelligence

Beyond identifying themes, hospital planning requires understanding
how specific diagnoses translate into actual surgical procedures.

In this section, we mine diagnosis–procedure pairs to quantify
which operations are most strongly associated with which diagnoses,
and how these relationships vary by urgency, specialty, anaesthesia,
and patient demographics.


# ✅ Section 8: Diagnosis → Procedure Pair Mining (Clinical Linkage Intelligence)

---

## 🎯 Purpose of This Section

Beyond themes and topics, theatre planning needs very specific linkage intelligence:

✅ What procedures are most linked to which diagnoses?

Examples:
- SDH → Burr holes / craniotomy
- Appendicitis → Appendectomy
- Fracture femur → ORIF / IM nail
- Ectopic pregnancy → salpingectomy / laparotomy
- NRFS → Caesarean section

This section builds a data-driven linkage table that shows:

- the most common procedure(s) for each diagnosis
- how concentrated the diagnosis is (one procedure dominates vs many)
- how urgency (EM/EL), anaesthesia type, specialty, and demographics vary by linkage

---

## ✅ Outputs We Will Produce

1) Top diagnosis → procedure pairs (overall burden)
2) For each diagnosis: top linked procedures + their share (%)
3) Specialty-specific linkage (optional)
4) Linkages stratified by Emergency vs Elective and Anaesthesia


In [25]:
def basic_clean(text):
    text = str(text).lower()
    text = text.replace("#", " fracture ")
    text = text.replace("&", " and ")
    text = text.replace("/", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Clean diagnosis and operations separately
df["diag_clean"] = df["DIAGNOSIS"].fillna("").apply(basic_clean)
df["proc_clean"] = df["OPERATIONS"].fillna("").apply(basic_clean)

# Quick check
df[["DIAGNOSIS", "OPERATIONS", "diag_clean", "proc_clean"]].head()


,DIAGNOSIS,OPERATIONS,diag_clean,proc_clean
0,CORNEAL LACERATION,CORNEAL REPAIR,corneal laceration,corneal repair
1,CORNEAL PERFORATION,CORNEAL REPAIR,corneal perforation,corneal repair
2,RIGHT ECTOPIC PREGNANCY,EXPLORATORY LAPAROTOMY/SALPINGECTOMY,right ectopic pregnancy,exploratory laparotomy salpingectomy
3,MULTIPLE FACIAL LACERATION,DEBRIDEMENT/STITCHING,multiple facial laceration,debridement stitching
4,COMPLICATED APPENDICITS,APPENDICITS,complicated appendicits,appendicits


In [26]:
pair_counts = (
    df.groupby(["diag_clean", "proc_clean"])
      .size()
      .reset_index(name="cases")
      .sort_values("cases", ascending=False)
)

pair_counts.head(20)


,diag_clean,proc_clean,cases
23981,non reassuring fetal status,caesarean section,1101
2956,adenotonsillar hypertrophy,adenotonsillectomy,852
4302,arrested dilatation,caesarean section,302
16073,hydrocephalus,ventriculo peritoneal shunt,290
1183,2 previous scar,caesarean section,246
99,1 previous scar,caesarean section,240
24022,non reassuring fetal status,emergency cesarean section,240
24729,obstructed labour,caesarean section,198
247,1 previous scar declined tolac,caesarean section,194
4221,arrested descent,caesarean section,177


In [27]:
# total cases per diagnosis
diag_totals = df.groupby("diag_clean").size().rename("diag_cases").reset_index()

# cases per diagnosis-procedure pair
diag_proc = (
    df.groupby(["diag_clean", "proc_clean"])
      .size()
      .rename("pair_cases")
      .reset_index()
)

# merge totals + compute share
diag_proc = diag_proc.merge(diag_totals, on="diag_clean", how="left")
diag_proc["pair_share_pct"] = (diag_proc["pair_cases"] / diag_proc["diag_cases"] * 100).round(2)

# rank procedures within each diagnosis
diag_proc["rank_within_diag"] = diag_proc.groupby("diag_clean")["pair_cases"].rank(method="first", ascending=False)

# show top 3 procedures per diagnosis (for the most common 20 diagnoses)
top_diag = diag_totals.sort_values("diag_cases", ascending=False).head(20)["diag_clean"]

top_linkages = (
    diag_proc[(diag_proc["diag_clean"].isin(top_diag)) & (diag_proc["rank_within_diag"] <= 3)]
    .sort_values(["diag_cases", "diag_clean", "rank_within_diag"], ascending=[False, True, True])
)

top_linkages.head(60)


,diag_clean,proc_clean,pair_cases,diag_cases,pair_share_pct,rank_within_diag
23981,non reassuring fetal status,caesarean section,1101,1637,67.26,1.0
24022,non reassuring fetal status,emergency cesarean section,240,1637,14.66,2.0
24014,non reassuring fetal status,emergency caesarean section,120,1637,7.33,3.0
2956,adenotonsillar hypertrophy,adenotonsillectomy,852,919,92.71,1.0
2952,adenotonsillar hypertrophy,adenotonsilectomy,21,919,2.29,2.0
2945,adenotonsillar hypertrophy,adenoidectomy,5,919,0.54,3.0
34367,schizophrenia,ect,75,818,9.17,1.0
34379,schizophrenia,electroconvulsive therapy,72,818,8.80,2.0
34369,schizophrenia,electro convulsive therapy,61,818,7.46,3.0
16073,hydrocephalus,ventriculo peritoneal shunt,290,590,49.15,1.0


In [28]:
# Take top 30 overall pairs
top_pairs = pair_counts.head(30)[["diag_clean", "proc_clean"]]

# Join back to original cases to summarize context for those top pairs
top_pair_cases = df.merge(top_pairs, on=["diag_clean", "proc_clean"], how="inner")

pair_context = top_pair_cases.groupby(["diag_clean", "proc_clean"]).agg(
    cases=("diag_clean", "size"),
    emergency_pct=("TYPE_OF_SURGERY", lambda x: (x=="EM").mean()*100),
    mean_age=("AGE_YEARS", "mean"),
    female_pct=("SEX", lambda x: (x.str.upper()=="FEMALE").mean()*100),
    top_specialty=("SPECIALTY", lambda x: x.value_counts().idxmax()),
    top_anaesthesia=("ANESTHESIA", lambda x: x.value_counts().idxmax())
).reset_index()

pair_context[["emergency_pct","mean_age","female_pct"]] = pair_context[["emergency_pct","mean_age","female_pct"]].round(2)

pair_context.sort_values("cases", ascending=False).head(30)

,diag_clean,proc_clean,cases,emergency_pct,mean_age,female_pct,top_specialty,top_anaesthesia
23,non reassuring fetal status,caesarean section,1101,99.27,26.18,100.00,OBS,SA
6,adenotonsillar hypertrophy,adenotonsillectomy,852,0.59,5.08,45.07,ENT,GA
9,arrested dilatation,caesarean section,302,99.01,25.64,100.00,OBS,SA
20,hydrocephalus,ventriculo peritoneal shunt,290,83.79,10.94,47.93,NEURO,GA
2,2 previous scar,caesarean section,246,40.24,32.05,100.00,OBS,SA
0,1 previous scar,caesarean section,240,47.08,29.66,100.00,OBS,SA
25,non reassuring fetal status,emergency cesarean section,240,99.58,26.11,100.00,OBSTETRICS,GA
27,obstructed labour,caesarean section,198,99.49,24.66,100.00,OBS,SA
1,1 previous scar declined tolac,caesarean section,194,85.05,28.41,100.00,OBS,SA
8,arrested descent,caesarean section,177,99.44,25.97,100.00,OBS,SA


# ✅ Section 9: Executive Synthesis – Clinical and Operational Insights

This section synthesizes findings from clustering, topic modeling,
and diagnosis–procedure linkage analysis into high-level clinical
and operational insights relevant for theatre planning and management.

### Key Observations

1. **Obstetric surgery is the dominant emergency driver**  
   Emergency Caesarean sections triggered by fetal distress,
   labour arrest, and repeat scar pregnancies represent the
   largest source of urgent theatre activation.

2. **Some surgical pathways are highly standardized**  
   Examples such as adenotonsillar hypertrophy → adenotonsillectomy
   show strong diagnosis–procedure concentration, making them ideal
   for structured elective scheduling.

3. **Other diagnoses demonstrate procedural complexity**  
   Conditions such as intestinal obstruction, urethral stricture,
   and neurosurgical hematomas map to multiple operative pathways,
   indicating higher planning complexity.

4. **Anaesthesia burden aligns strongly with clinical themes**  
   Obstetric emergencies are predominantly spinal-based,
   while trauma, infection, and neurosurgical cases are
   general-anaesthesia intensive.

5. **Terminology variation impacts analytics quality**  
   Multiple spelling and naming variants were observed for the same
   procedures (e.g. appendectomy, Caesarean section), highlighting
   the importance of clinical terminology standardization.

These insights demonstrate how Clinical NLP can transform free-text
theatre records into actionable intelligence for capacity planning,
staffing, and resource allocation.


# ✅ Section 10: Exporting NLP Outputs for Reporting and Dashboards

---

## 🎯 Purpose of This Section

After completing clustering, topic modeling, and diagnosis–procedure linkage mining,
we export the key executive outputs into structured CSV files.

These exports allow:

- dashboard building (Power BI / Tableau)
- reporting for theatre leadership
- reproducible analysis for GitHub
- hospital planning documentation

---

## ✅ Files Exported

1. Theme Cluster Summary  
2. Top Terms per Cluster  
3. Top Terms per Topic (LDA)  
4. Top Diagnosis–Procedure Pairs  
5. Diagnosis → Top Procedures + Share (%)  
6. Executive Pair Burden Context Table


In [29]:
import os

export_path = "clinical_nlp_exports"
os.makedirs(export_path, exist_ok=True)

print("Export folder created:", export_path)


Export folder created: clinical_nlp_exports


In [30]:
theme_summary.to_csv(f"{export_path}/theme_cluster_summary.csv")
print("✅ Saved: theme_cluster_summary.csv")


✅ Saved: theme_cluster_summary.csv


In [31]:
import pandas as pd
import numpy as np

cluster_term_rows = []

terms = tfidf.get_feature_names_out()

for i in range(k):
    center_terms = kmeans.cluster_centers_[i].argsort()[-15:][::-1]
    top_words = [terms[j] for j in center_terms]

    cluster_term_rows.append({
        "cluster_id": i,
        "top_terms": ", ".join(top_words)
    })

cluster_terms_df = pd.DataFrame(cluster_term_rows)

cluster_terms_df.to_csv(f"{export_path}/cluster_top_terms.csv", index=False)
print("✅ Saved: cluster_top_terms.csv")

cluster_terms_df

✅ Saved: cluster_top_terms.csv


,cluster_id,top_terms
0,0,"right, left, repair, laparotomy, excision, cra..."
1,1,"fracture, open, fixation, reduction, open redu..."
2,2,"therapy, electroconvulsive, electroconvulsive ..."
3,3,"section, caesarean, caesarean section, scar, p..."
4,4,"cataract, eye, lens, intraocular, small, small..."
5,5,"reassuring, non reassuring, reassuring fetal, ..."
6,6,"debridement, wound, skin, graft, skin graft, t..."
7,7,"adenotonsillectomy, adenotonsillar, hypertroph..."


In [32]:
topic_rows = []

topic_terms = count_vec.get_feature_names_out()

for topic_idx, topic in enumerate(lda13.components_):
    top_features = topic.argsort()[-15:][::-1]
    top_words = [topic_terms[i] for i in top_features]

    topic_rows.append({
        "topic_id": topic_idx,
        "top_terms": ", ".join(top_words)
    })

topic_terms_df = pd.DataFrame(topic_rows)

topic_terms_df.to_csv(f"{export_path}/lda13_topic_terms.csv", index=False)
print("✅ Saved: lda13_topic_terms.csv")

topic_terms_df

✅ Saved: lda13_topic_terms.csv


,topic_id,top_terms
0,0,"section, caesarean, caesarean section, fetal, ..."
1,1,"eye, cataract, left, right, left eye, right ey..."
2,2,"skin, graft, thickness, skin graft, drainage, ..."
3,3,"debridement, amputation, right, left, cranioto..."
4,4,"section, caesarean, caesarean section, scar, p..."
5,5,"hypertrophy, adenotonsillectomy, adenotonsilla..."
6,6,"fracture, right, debridement, left, tibia, ope..."
7,7,"laparotomy, exploratory, exploratory laparotom..."
8,8,"fracture, open, reduction, open reduction, int..."
9,9,"tear, repair, examination, anesthesia, examina..."


In [33]:
pair_counts.head(300).to_csv(f"{export_path}/top_diagnosis_procedure_pairs.csv", index=False)
print("✅ Saved: top_diagnosis_procedure_pairs.csv")

✅ Saved: top_diagnosis_procedure_pairs.csv


In [34]:
top_linkages.to_csv(f"{export_path}/diagnosis_top_procedure_linkages.csv", index=False)
print("✅ Saved: diagnosis_top_procedure_linkages.csv")


✅ Saved: diagnosis_top_procedure_linkages.csv


In [35]:
pair_context.to_csv(f"{export_path}/executive_pair_context_table.csv", index=False)
print("✅ Saved: executive_pair_context_table.csv")


✅ Saved: executive_pair_context_table.csv


In [36]:
os.listdir(export_path)


['cluster_label_lookup.csv',
 'cluster_top_terms.csv',
 'diagnosis_top_procedure_linkages.csv',
 'executive_pair_context_table.csv',
 'lda13_topic_terms.csv',
 'theme_cluster_summary.csv',
 'topic_label_lookup.csv',
 'top_diagnosis_procedure_pairs.csv']

In [37]:
# --- Cluster labels from your interpretation ---
cluster_labels = {
    0: "General Major Surgery & Neurosurgical Core Burden",
    1: "Trauma & Orthopaedic Fracture Fixation (ORIF)",
    2: "Electroconvulsive Therapy (ECT) Cluster",
    3: "Obstetric Caesarean Burden (Scar/Labour Arrest/Breech)",
    4: "Ophthalmology Cataract & Lens Surgery",
    5: "OB Emergency Fetal Distress (NRFS) Caesarean",
    6: "Infection/Wounds: Debridement + Skin Graft",
    7: "ENT Paediatric Airway: Adenotonsillectomy + OSA"
}

df["theme_cluster_label"] = df["theme_cluster"].map(cluster_labels)

# Export a small lookup table for dashboards
import pandas as pd
cluster_lookup = pd.DataFrame(
    [{"theme_cluster": k, "theme_cluster_label": v} for k, v in cluster_labels.items()]
)
cluster_lookup.to_csv(f"{export_path}/cluster_label_lookup.csv", index=False)

print("✅ Saved: cluster_label_lookup.csv")
cluster_lookup


✅ Saved: cluster_label_lookup.csv


,theme_cluster,theme_cluster_label
0,0,General Major Surgery & Neurosurgical Core Burden
1,1,Trauma & Orthopaedic Fracture Fixation (ORIF)
2,2,Electroconvulsive Therapy (ECT) Cluster
3,3,Obstetric Caesarean Burden (Scar/Labour Arrest...
4,4,Ophthalmology Cataract & Lens Surgery
5,5,OB Emergency Fetal Distress (NRFS) Caesarean
6,6,Infection/Wounds: Debridement + Skin Graft
7,7,ENT Paediatric Airway: Adenotonsillectomy + OSA


In [38]:
import shutil

zip_name = "clinical_nlp_exports.zip"
shutil.make_archive("clinical_nlp_exports", "zip", export_path)

zip_name


'clinical_nlp_exports.zip'

In [39]:
import pandas as pd

# Cluster labels (from our clinical interpretation)
cluster_labels = {
    0: "General Major Surgery & Neurosurgical Core Burden",
    1: "Trauma & Orthopaedic Fracture Fixation (ORIF)",
    2: "Electroconvulsive Therapy (ECT) Cluster",
    3: "Obstetric Caesarean Burden (Scar/Labour Arrest/Breech)",
    4: "Ophthalmology Cataract & Lens Surgery",
    5: "OB Emergency Fetal Distress (NRFS) Caesarean",
    6: "Infection/Wounds: Debridement + Skin Graft",
    7: "ENT Paediatric Airway: Adenotonsillectomy + OSA"
}

# Create lookup table
cluster_lookup = pd.DataFrame(
    [{"theme_cluster": k, "theme_cluster_label": v} for k, v in cluster_labels.items()]
)

# Save for dashboard joins
cluster_lookup.to_csv(f"{export_path}/cluster_label_lookup.csv", index=False)

print("✅ Saved: cluster_label_lookup.csv")
cluster_lookup


✅ Saved: cluster_label_lookup.csv


,theme_cluster,theme_cluster_label
0,0,General Major Surgery & Neurosurgical Core Burden
1,1,Trauma & Orthopaedic Fracture Fixation (ORIF)
2,2,Electroconvulsive Therapy (ECT) Cluster
3,3,Obstetric Caesarean Burden (Scar/Labour Arrest...
4,4,Ophthalmology Cataract & Lens Surgery
5,5,OB Emergency Fetal Distress (NRFS) Caesarean
6,6,Infection/Wounds: Debridement + Skin Graft
7,7,ENT Paediatric Airway: Adenotonsillectomy + OSA


In [40]:
topic_labels = {
    0: "OB Emergency Fetal Distress Caesarean",
    1: "Ophthalmology Cataract & Corneal Surgery",
    2: "Abscess Drainage + Skin Graft Infection Theme",
    3: "Severe Trauma + Amputation + Neuro Injury",
    4: "Repeat Caesarean Scar + TOLAC Complexity",
    5: "ENT Airway + Foreign Body Procedures",
    6: "Orthopaedics Mixed Trauma + Arthroplasty",
    7: "General Surgery Laparotomy + Bowel Obstruction",
    8: "Orthopaedic ORIF Fixation Theme",
    9: "Perineal Tears + EUA + Hemorrhage",
    10: "Neurosurgery Chronic Subdural Hematoma Burr Holes",
    11: "Ectopic Pregnancy + Tumor Excision Laparotomy",
    12: "Mixed Elective Procedures (ECT/Hernia/Hysterectomy)"
}

topic_lookup = pd.DataFrame(
    [{"topic_id": k, "topic_label": v} for k, v in topic_labels.items()]
)

topic_lookup.to_csv(f"{export_path}/topic_label_lookup.csv", index=False)

print("✅ Saved: topic_label_lookup.csv")
topic_lookup


✅ Saved: topic_label_lookup.csv


,topic_id,topic_label
0,0,OB Emergency Fetal Distress Caesarean
1,1,Ophthalmology Cataract & Corneal Surgery
2,2,Abscess Drainage + Skin Graft Infection Theme
3,3,Severe Trauma + Amputation + Neuro Injury
4,4,Repeat Caesarean Scar + TOLAC Complexity
5,5,ENT Airway + Foreign Body Procedures
6,6,Orthopaedics Mixed Trauma + Arthroplasty
7,7,General Surgery Laparotomy + Bowel Obstruction
8,8,Orthopaedic ORIF Fixation Theme
9,9,Perineal Tears + EUA + Hemorrhage


In [41]:
import shutil

# Create zip archive of all exports
shutil.make_archive("clinical_nlp_exports", "zip", export_path)

print("✅ ZIP file created: clinical_nlp_exports.zip")


✅ ZIP file created: clinical_nlp_exports.zip


In [42]:
os.listdir(export_path)


['cluster_label_lookup.csv',
 'cluster_top_terms.csv',
 'diagnosis_top_procedure_linkages.csv',
 'executive_pair_context_table.csv',
 'lda13_topic_terms.csv',
 'theme_cluster_summary.csv',
 'topic_label_lookup.csv',
 'top_diagnosis_procedure_pairs.csv']

# ✅ Section 11: Conclusion and Executive Summary

### Project Summary

This project applied advanced Clinical Natural Language Processing (NLP)
techniques to theatre operation data from 2022–2025 in order to uncover
surgical burden patterns, case complexity, and diagnosis–procedure
relationships.

Rather than relying on simple word counts, the analysis implemented a
full real-world healthcare NLP workflow, including clinical text cleaning,
TF-IDF feature engineering, unsupervised clustering, topic modeling,
and diagnosis–procedure linkage mining.

### Key Outcomes

- Identification of dominant surgical burden themes across specialties
- Quantification of emergency vs elective drivers of theatre workload
- Mapping of diagnoses to their most common operative procedures
- Stratification of surgical burden by age, sex, specialty, and anaesthesia
- Generation of executive-ready, dashboard-friendly output tables

### Operational Value

The findings support data-driven decision-making in:
- theatre capacity planning
- emergency preparedness
- anaesthesia staffing
- elective scheduling optimization
- clinical documentation standardization

### Final Note

This work demonstrates how Clinical NLP can unlock high-value insights
from routinely collected hospital text data, enabling health systems
to better understand and manage surgical complexity at scale.
